In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M25.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7706053690933775, 'n_it': 0.39797030117648785}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[17.549992003065007, 13.922283910564314, 13.879815009331775, 16.537345337650855, 14.414551870858839, 13.931752643011876, 13.782718994592189, 15.678601447385693, 13.270962557243998, 13.336874498783608, 14.07890924682157, 13.691457271111952, 17.59466750598057, 17.464524569753276, 15.117075364218552, 13.597859259527022, 15.155389059903534, 16.508100814523125, 17.177582271483974, 13.65207903379294, 14.54938777035031, 17.583740086635746, 13.970050858762509, 13.83106662323986, 18.15636840908683, 14.538341170136196, 13.841222184080497, 13.437016846656755, 14.05681909570306, 13.599439481359157, 17.254309032047836, 14.444346582648018, 14.119109205419527, 16.88778225142387, 13.332383318216921, 14.641002043752136, 16.85938016880509, 14.767898289491598, 15.253916938989121, 14.937398706229537, 13.475382891514139, 14.394173609890167, 13.493025047289052, 13.731930480988296, 13.76098574423234, 13.839422056088864, 14.028796856803218, 14.693338477240447, 13.652345891768292, 13.856946394753141, 14.081329

In [5]:
np.average(y_max_arr)

np.float64(14.978029411825952)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)